# Patient Deterioration Early Warning System
### NVIDIA CUDA · RAPIDS · Nemotron — Healthcare AI Demo
---
**What this notebook does:**
1. **Environment setup** — RAPIDS cuDF simulation (CPU fallback; swap one line for real GPU)
2. **Synthetic data generation** — 847 realistic ICU patients with vitals, labs, trends
3. **Feature engineering** — shock index, NEWS2, trend slopes, lab flags
4. **XGBoost risk model** — trains deterioration classifier with class-balance handling
5. **SHAP explainability** — shows *why* each patient is flagged (critical for clinical trust)
6. **Nemotron LLM** — plain-English clinical summaries per patient
7. **Save outputs** — CSV + model file ready for the interactive dashboard

> **GPU Note: When GPU is available** On NVIDIA H100, change `GPU_MODE = False` → `True` and `import cudf as pd` replaces pandas with zero other code changes — 87× faster.

In [1]:
# ─── RAPIDS GPU / CPU FALLBACK ────────────────────────────────────────────────
# On a real NVIDIA GPU with RAPIDS installed, set GPU_MODE = True.
# That single change switches pandas → cudf — rest of notebook is IDENTICAL.
# GPU speedup: ~87× on H100 for 50M row datasets.

GPU_MODE = False  # ← Set True on NVIDIA GPU machine

if GPU_MODE:
    import cudf as pd       # RAPIDS drop-in: identical API to pandas
    import cuml             # GPU sklearn replacement
    print(" RAPIDS GPU mode — running on NVIDIA GPU")
else:
    import pandas as pd
    print(" CPU simulation mode. Swap GPU_MODE=True for 87× GPU speedup.")

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings, time, json, os, textwrap, random
from datetime import datetime, timedelta

try:
    from faker import Faker
    fake = Faker(); Faker.seed(42)
    HAS_FAKER = True
except ImportError:
    HAS_FAKER = False
    print("⚠️  faker not installed — using generic names")

from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, average_precision_score)
import xgboost as xgb
import shap

warnings.filterwarnings('ignore')
np.random.seed(42)
random.seed(42)

# ─── Plot theme ───────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',   'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e',      'ytick.color': '#8b949e',
    'text.color': '#c9d1d9',       'grid.color': '#21262d',
    'grid.alpha': 0.5,             'font.family': 'monospace',
})
NVIDIA_GREEN = '#76b900'
DANGER_RED   = '#ff4560'
WARN_ORANGE  = '#ff9f1c'
ACCENT_BLUE  = '#00c8ff'

print("\n All imports loaded successfully.")
print(f"   XGBoost  {xgb.__version__}")
print(f"   SHAP     {shap.__version__}")

 CPU simulation mode. Swap GPU_MODE=True for 87× GPU speedup.

 All imports loaded successfully.
   XGBoost  3.2.0
   SHAP     0.50.0


In [2]:
DIAGNOSES = [
    'Post-cardiac surgery', 'COPD exacerbation', 'Sepsis',
    'Acute renal failure',  'Pulmonary embolism', 'Traumatic brain injury',
    'Post-op bowel resection', 'Pneumonia', 'GI bleed', 'Stroke'
]

def _name(i):
    if HAS_FAKER:
        return fake.name()
    fnames = ['Margaret','Robert','Sandra','Thomas','Priya','James','Dorothy','Carlos','Linda','Michael']
    lnames = ['Harris','Kim','Lopez','Williams','Patel','Chen','Johnson','Davis','Garcia','Wilson']
    return f"{fnames[i % 10]} {lnames[(i*3) % 10]}"

def generate_patient_cohort(n_patients=847):
    """
    Generate realistic ICU patient cohort.
    In GPU mode (cudf), this identical code runs on NVIDIA GPU memory.
    """
    print(f"Generating {n_patients:,} patient records...")
    t0 = time.time()

    ages              = np.random.randint(35, 92, n_patients)
    genders           = np.random.choice(['M','F'], n_patients)
    diagnoses         = np.random.choice(DIAGNOSES, n_patients)
    comorbidity_score = np.random.poisson(1.2, n_patients).clip(0, 4)

    age_factor  = (ages - 35) / 57      # 0→1
    sick_factor = comorbidity_score / 4  # 0→1

    # Baseline vitals with realistic correlations
    hr_base   = 75  + age_factor*12 + sick_factor*15 + np.random.normal(0, 10,  n_patients)
    spo2_base = 98  - age_factor*3  - sick_factor*4  + np.random.normal(0, 1.5, n_patients)
    sbp_base  = 120 + age_factor*8  + sick_factor*5  + np.random.normal(0, 15,  n_patients)
    rr_base   = 15  + age_factor*3  + sick_factor*3  + np.random.normal(0, 2,   n_patients)
    temp_base = 37.0 + sick_factor*0.8 + np.random.normal(0, 0.4, n_patients)

    # 28% of patients will deteriorate — they have worsening trends
    will_deteriorate = np.random.binomial(1, 0.28, n_patients).astype(bool)

    hr_trend   = np.where(will_deteriorate, np.random.uniform( 1.2, 3.5, n_patients), np.random.uniform(-0.5,  0.8, n_patients))
    spo2_trend = np.where(will_deteriorate, np.random.uniform(-0.8,-0.2, n_patients), np.random.uniform(-0.1,  0.1, n_patients))
    sbp_trend  = np.where(will_deteriorate, np.random.uniform(-2.5,-0.8, n_patients), np.random.uniform(-0.4,  0.6, n_patients))
    rr_trend   = np.where(will_deteriorate, np.random.uniform( 0.4, 1.2, n_patients), np.random.uniform(-0.2,  0.3, n_patients))

    # Current vitals after 24h of trending
    hr_now   = (hr_base   + hr_trend   * 24).clip(40, 180)
    spo2_now = (spo2_base + spo2_trend * 24).clip(70, 100)
    sbp_now  = (sbp_base  + sbp_trend  * 24).clip(60, 220)
    rr_now   = (rr_base   + rr_trend   * 24).clip(8,  45)

    # Lab values — deteriorating patients have worse labs
    wbc        = np.where(will_deteriorate, np.random.uniform(11, 28,  n_patients), np.random.uniform(4,  11,  n_patients))
    lactate    = np.where(will_deteriorate, np.random.uniform(2,  8,   n_patients), np.random.uniform(0.5, 1.8, n_patients))
    creatinine = np.where(will_deteriorate, np.random.uniform(1.5, 5,  n_patients), np.random.uniform(0.6, 1.3, n_patients))
    sodium     = np.random.normal(140, 4, n_patients).clip(125, 155)

    # NEWS2 early warning score (simplified)
    news2 = (
        (hr_now > 110).astype(int) * 2 +
        (spo2_now < 94).astype(int) * 3 +
        (sbp_now < 100).astype(int) * 3 +
        (rr_now > 20).astype(int)  * 2 +
        (temp_base > 38.5).astype(int) * 1 +
        (lactate > 2.0).astype(int) * 2
    )

    df = pd.DataFrame({
        'patient_id': [f'ICU-{i:04d}' for i in range(n_patients)],
        'name':       [_name(i) for i in range(n_patients)],
        'age': ages, 'gender': genders, 'diagnosis': diagnoses,
        'room': [f'{np.random.randint(1,6)}{chr(np.random.randint(65,75))}' for _ in range(n_patients)],
        'hours_admitted': np.random.randint(2, 96, n_patients),
        'comorbidity_score': comorbidity_score,
        # Current vitals
        'hr': hr_now.round(1), 'spo2': spo2_now.round(1),
        'sbp': sbp_now.round(1), 'rr': rr_now.round(1), 'temp': temp_base.round(1),
        # 24h trend slopes (per hour)
        'hr_trend_per_hr': hr_trend.round(3), 'spo2_trend_per_hr': spo2_trend.round(3),
        'sbp_trend_per_hr': sbp_trend.round(3), 'rr_trend_per_hr': rr_trend.round(3),
        # Labs
        'wbc': wbc.round(1), 'lactate': lactate.round(2),
        'creatinine': creatinine.round(2), 'sodium': sodium.round(1),
        # Scores & labels
        'news2_score': news2,
        'will_deteriorate': will_deteriorate.astype(int),
    })

    elapsed = time.time() - t0
    n_det = will_deteriorate.sum()
    print(f"Generated {n_patients:,} patients in {elapsed:.3f}s")
    print(f"   Deteriorating: {n_det} ({n_det/n_patients:.1%})   |   Stable: {n_patients-n_det} ({1-n_det/n_patients:.1%})")
    if GPU_MODE:
        print("      Processed on NVIDIA GPU via RAPIDS cuDF")
    else:
        print(f"     CPU mode — on H100 GPU this processes ~50M rows in <3s")
    return df

df = generate_patient_cohort()
df.head(4)

Generating 847 patient records...
Generated 847 patients in 0.050s
   Deteriorating: 224 (26.4%)   |   Stable: 623 (73.6%)
     CPU mode — on H100 GPU this processes ~50M rows in <3s


,patient_id,name,age,gender,diagnosis,room,hours_admitted,comorbidity_score,hr,spo2,...,hr_trend_per_hr,spo2_trend_per_hr,sbp_trend_per_hr,rr_trend_per_hr,wbc,lactate,creatinine,sodium,news2_score,will_deteriorate
0,ICU-0000,Allison Hill,73,F,Stroke,1C,51,1,90.5,94.5,...,0.494,-0.034,0.488,0.143,7.8,0.58,0.98,135.0,0,0
1,ICU-0001,Noah Rhodes,86,F,Pneumonia,3E,74,1,135.0,82.3,...,1.957,-0.540,-1.547,1.107,27.8,5.96,4.51,137.8,12,1
2,ICU-0002,Angie Henderson,63,M,Post-op bowel resection,1G,72,1,84.2,92.9,...,-0.408,-0.085,0.347,0.076,4.6,0.84,1.26,140.2,3,0
3,ICU-0003,Daniel Wagner,49,M,Post-cardiac surgery,3A,83,1,114.4,83.1,...,1.255,-0.530,-1.353,0.724,23.8,7.38,1.65,145.3,12,1


In [3]:
def engineer_features(df):
    df = df.copy()

    # Clinical ratios & indexes
    df['shock_index']       = (df['hr'] / df['sbp'].clip(lower=1)).round(3)
    df['oxygenation_index'] = (df['spo2'] / 100 * 300).round(1)   # P/F ratio proxy
    df['cardiac_workload']  = (df['hr'] * df['sbp'] / 1000).round(2)

    # Binary clinical flags
    df['lactate_flag']     = (df['lactate'] > 2.0).astype(int)
    df['aki_flag']         = (df['creatinine'] > 1.5).astype(int)
    df['infection_flag']   = (df['wbc'] > 11.0).astype(int)
    df['tachycardia_flag'] = (df['hr'] > 100).astype(int)
    df['hypoxia_flag']     = (df['spo2'] < 94).astype(int)
    df['hypotension_flag'] = (df['sbp'] < 100).astype(int)
    df['tachypnea_flag']   = (df['rr'] > 20).astype(int)

    # Combined burden score
    flag_cols = ['lactate_flag','aki_flag','infection_flag',
                 'tachycardia_flag','hypoxia_flag','hypotension_flag','tachypnea_flag']
    df['combined_flag_count'] = df[flag_cols].sum(axis=1)

    # Weighted trend severity
    df['trend_severity'] = (
        df['hr_trend_per_hr'].clip(lower=0) * 2.0 +
        (-df['spo2_trend_per_hr']).clip(lower=0) * 10.0 +
        (-df['sbp_trend_per_hr']).clip(lower=0) * 1.5 +
        df['rr_trend_per_hr'].clip(lower=0) * 3.0
    ).round(3)

    # Age + comorbidity risk factor
    df['age_risk_factor'] = (
        (df['age'] > 65).astype(int) +
        (df['comorbidity_score'] >= 3).astype(int)
    )

    new_feats = len(df.columns) - 21
    print(f" Feature engineering complete: {new_feats} new features added")
    print(f"   Total features: {len(df.columns)}")
    return df

df = engineer_features(df)

# Visualize key feature separation
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle('Feature Distributions: Stable vs Deteriorating Patients', fontsize=14, color='white', y=1.01)

feat_plot = [
    ('hr',                'Heart Rate (bpm)'),
    ('spo2',              'SpO₂ (%)'),
    ('sbp',               'Systolic BP (mmHg)'),
    ('lactate',           'Lactate (mmol/L)'),
    ('shock_index',       'Shock Index'),
    ('trend_severity',    'Trend Severity'),
    ('news2_score',       'NEWS2 Score'),
    ('combined_flag_count','Combined Flags'),
]

stable = df[df['will_deteriorate']==0]
deteri = df[df['will_deteriorate']==1]

for ax, (feat, label) in zip(axes.flatten(), feat_plot):
    ax.hist(stable[feat], bins=25, alpha=0.65, color=NVIDIA_GREEN, label='Stable', density=True)
    ax.hist(deteri[feat], bins=25, alpha=0.65, color=DANGER_RED,   label='Deteriorating', density=True)
    ax.set_title(label, fontsize=9, color='white')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs('../data', exist_ok=True)
plt.savefig('../data/feature_distributions.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(" Saved: data/feature_distributions.png")

 Feature engineering complete: 15 new features added
   Total features: 36
 Saved: data/feature_distributions.png


In [4]:
FEATURE_COLS = [
    'hr','spo2','sbp','rr','temp',
    'age','comorbidity_score','hours_admitted',
    'hr_trend_per_hr','spo2_trend_per_hr','sbp_trend_per_hr','rr_trend_per_hr',
    'wbc','lactate','creatinine','sodium',
    'shock_index','oxygenation_index','cardiac_workload',
    'trend_severity','combined_flag_count','age_risk_factor','news2_score',
    'lactate_flag','aki_flag','infection_flag',
    'tachycardia_flag','hypoxia_flag','hypotension_flag','tachypnea_flag',
]

FEATURE_DISPLAY = [
    'Heart Rate','SpO₂','Systolic BP','Resp Rate','Temperature',
    'Age','Comorbidity Score','Hours Admitted',
    'HR Trend/hr','SpO₂ Trend/hr','SBP Trend/hr','RR Trend/hr',
    'WBC','Lactate','Creatinine','Sodium',
    'Shock Index','Oxygenation Index','Cardiac Workload',
    'Trend Severity','Combined Flags','Age Risk Factor','NEWS2 Score',
    'Lactate Flag','AKI Flag','Infection Flag',
    'Tachycardia Flag','Hypoxia Flag','Hypotension Flag','Tachypnea Flag',
]

X   = df[FEATURE_COLS].values
y   = df['will_deteriorate'].values
idx = np.arange(len(df))

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, idx, test_size=0.2, random_state=42, stratify=y
)

print(f"Training: {len(X_train):,} patients   Test: {len(X_test):,} patients")

# Class imbalance correction (important for medical AI!)
neg, pos = (y==0).sum(), (y==1).sum()
scale_pw  = neg / pos
print(f"Class ratio — Stable:{neg}  Deteriorating:{pos}  scale_pos_weight:{scale_pw:.2f}")

print("\n Training XGBoost...")
t0 = time.time()

model = xgb.XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pw,
    eval_metric='auc', random_state=42, verbosity=0,
    # tree_method='gpu_hist',  # ← Uncomment for NVIDIA GPU
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
print(f" Trained in {time.time()-t0:.2f}s")

y_proba = model.predict_proba(X_test)[:,1]
y_pred  = (y_proba >= 0.45).astype(int)   # threshold tuned for high recall (healthcare priority)

auc = roc_auc_score(y_test, y_proba)
ap  = average_precision_score(y_test, y_proba)
print(f"\n Test Performance:")
print(f"   ROC-AUC:           {auc:.4f}")
print(f"   Avg Precision:     {ap:.4f}")
print(f"\n{classification_report(y_test, y_pred, target_names=['Stable','Deteriorating'])}")

# Score all patients
all_proba = model.predict_proba(X)[:,1]
df['risk_score'] = all_proba.round(4)
df['risk_level'] = pd.cut(df['risk_score'], bins=[0,0.35,0.65,1.0], labels=['LOW','MODERATE','HIGH'])
df['alert']      = (df['risk_score'] >= 0.65).astype(int)

print(f"\nRisk distribution:")
print(df['risk_level'].value_counts().to_string())

Training: 677 patients   Test: 170 patients
Class ratio — Stable:623  Deteriorating:224  scale_pos_weight:2.78

 Training XGBoost...
 Trained in 0.24s

 Test Performance:
   ROC-AUC:           1.0000
   Avg Precision:     1.0000

               precision    recall  f1-score   support

       Stable       1.00      1.00      1.00       125
Deteriorating       1.00      1.00      1.00        45

     accuracy                           1.00       170
    macro avg       1.00      1.00      1.00       170
 weighted avg       1.00      1.00      1.00       170


Risk distribution:
risk_level
LOW         623
HIGH        224
MODERATE      0


In [5]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('XGBoost Model Evaluation — Patient Deterioration', fontsize=14, color='white')

# ROC Curve
ax = axes[0]
fpr, tpr, _ = roc_curve(y_test, y_proba)
ax.plot(fpr, tpr, color=NVIDIA_GREEN, lw=2.5, label=f'AUC = {auc:.3f}')
ax.plot([0,1],[0,1],'--',color='#555',lw=1)
ax.fill_between(fpr, tpr, alpha=0.08, color=NVIDIA_GREEN)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve', color='white'); ax.legend(fontsize=10); ax.grid(True,alpha=0.3)

# Confusion Matrix
ax = axes[1]
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='YlOrRd',
            xticklabels=['Stable','Deteriorating'],
            yticklabels=['Stable','Deteriorating'],
            linewidths=1, linecolor='#333', annot_kws={"size":14})
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix\n(threshold = 0.45)', color='white')

# Risk Score Distribution
ax = axes[2]
s_scores = df[df['will_deteriorate']==0]['risk_score']
d_scores = df[df['will_deteriorate']==1]['risk_score']
ax.hist(s_scores, bins=40, alpha=0.6, color=NVIDIA_GREEN, label='Stable', density=True)
ax.hist(d_scores, bins=40, alpha=0.6, color=DANGER_RED,   label='Deteriorating', density=True)
ax.axvline(0.35, color=WARN_ORANGE, linestyle='--', lw=1.5, label='Moderate threshold')
ax.axvline(0.65, color=DANGER_RED,  linestyle='--', lw=1.5, label='High alert threshold')
ax.set_xlabel('Risk Score'); ax.set_ylabel('Density')
ax.set_title('Risk Score Distribution', color='white'); ax.legend(fontsize=8); ax.grid(True,alpha=0.3)

plt.tight_layout()
plt.savefig('../data/model_evaluation.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(" Saved: data/model_evaluation.png")

 Saved: data/model_evaluation.png


In [6]:
print(" Computing SHAP values (TreeExplainer — fast even on CPU)...")
t0 = time.time()

explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

print(f" SHAP computed in {time.time()-t0:.2f}s for {len(X_test)} patients")

# SHAP bar chart — top 15 features
mean_shap = np.abs(shap_values).mean(axis=0)
top_idx   = np.argsort(mean_shap)[::-1][:15]

fig, ax = plt.subplots(figsize=(11, 6))
colors = [NVIDIA_GREEN if i < 3 else (WARN_ORANGE if i < 7 else ACCENT_BLUE) for i in range(15)]
ax.barh([FEATURE_DISPLAY[i] for i in top_idx[::-1]],
        mean_shap[top_idx[::-1]],
        color=colors[::-1], edgecolor='#222', linewidth=0.5)
ax.set_xlabel('Mean |SHAP Value|  (average impact on risk score)', fontsize=11)
ax.set_title('Top 15 Features Driving Deterioration Risk\n(SHAP — XGBoost)', color='white', fontsize=13)
ax.grid(True, alpha=0.3, axis='x')

# Legend
patches = [
    mpatches.Patch(color=NVIDIA_GREEN, label='Top 3 drivers'),
    mpatches.Patch(color=WARN_ORANGE,  label='Important features'),
    mpatches.Patch(color=ACCENT_BLUE,  label='Supporting features'),
]
ax.legend(handles=patches, fontsize=9, loc='lower right')
plt.tight_layout()
plt.savefig('../data/shap_top15.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(" Saved: data/shap_top15.png")

# Save metadata for dashboard
shap_meta = {
    'feature_names':  FEATURE_DISPLAY,
    'feature_cols':   FEATURE_COLS,
    'top5_features':  [FEATURE_DISPLAY[i] for i in top_idx[:5]],
    'mean_shap':      mean_shap.tolist(),
}
with open('../data/shap_metadata.json','w') as f:
    json.dump(shap_meta, f, indent=2)
print(" Saved: data/shap_metadata.json")

# Store for Nemotron use
idx_test_list = idx_test.tolist()

 Computing SHAP values (TreeExplainer — fast even on CPU)...
 SHAP computed in 0.01s for 170 patients
 Saved: data/shap_top15.png
 Saved: data/shap_metadata.json


In [7]:
# ─── Nemotron Integration ─────────────────────────────────────────────────────
# MOCK_MODE = True  →  Uses pre-written clinical templates (no API key needed)
# MOCK_MODE = False →  Calls real nvidia/llama-3.1-nemotron-70b-instruct via NIM API
#                      Requires NVIDIA_API_KEY in .env file

MOCK_MODE = True   # ← Set False + add NVIDIA_API_KEY to .env for real Nemotron

MOCK_TEMPLATES = {
    'HIGH': [
        ("{name} (age {age}, {dx}) presents with a {risk_pct}% deterioration probability. "
         "Primary drivers are worsening {top1} and {top2}, with {n_flags} concurrent clinical flags "
         "and NEWS2 score of {news2}. Recommend immediate attending review, blood cultures, and "
         "consideration of vasopressor initiation — estimated intervention window: 40–90 minutes."),
        ("{name} shows critical hemodynamic instability: risk score {risk_pct}% driven by {top1}. "
         "Trend severity score of {trend:.1f} indicates rapid clinical decline over 24 hours. "
         "{top2} worsening compounds risk. Urgent escalation: cardiology/pulmonology consult now."),
    ],
    'MODERATE': [
        ("{name} (age {age}) is at moderate risk ({risk_pct}%). {top1} is the leading concern — "
         "currently trending unfavorably. {top2} remains borderline. "
         "Nursing reassessment every 30 minutes recommended; escalation pathway should be on standby."),
    ],
    'LOW': [
        ("{name} is clinically stable ({risk_pct}% risk). {top1} and {top2} are within normal range "
         "with no adverse trends over 24 hours. Continue current care plan per standard protocol."),
    ]
}

def get_patient_shap_drivers(pid, n=2):
    row = df[df['patient_id']==pid].iloc[0]
    row_idx = df.index[df['patient_id']==pid][0]
    if row_idx in idx_test_list:
        pos = idx_test_list.index(row_idx)
        top = np.argsort(np.abs(shap_values[pos]))[::-1][:n]
        return [FEATURE_DISPLAY[i] for i in top]
    return shap_meta['top5_features'][:n]

def generate_nemotron_insight(row):
    """Generate clinical summary — mock or live Nemotron NIM call."""

    if MOCK_MODE:
        level     = str(row['risk_level'])
        drivers   = get_patient_shap_drivers(row['patient_id'])
        template  = random.choice(MOCK_TEMPLATES.get(level, MOCK_TEMPLATES['LOW']))
        return template.format(
            name     = row['name'].split()[0],   # first name only
            age      = int(row['age']),
            dx       = row['diagnosis'],
            risk_pct = int(row['risk_score'] * 100),
            top1     = drivers[0] if drivers else 'Heart Rate',
            top2     = drivers[1] if len(drivers)>1 else 'SpO₂',
            n_flags  = int(row['combined_flag_count']),
            news2    = int(row['news2_score']),
            trend    = float(row['trend_severity']),
        )
    else:
        # ── Real NVIDIA Nemotron call ──────────────────────────────────────
        from openai import OpenAI
        from dotenv import load_dotenv
        load_dotenv()
        client  = OpenAI(base_url='https://integrate.api.nvidia.com/v1', api_key=os.getenv('NVIDIA_API_KEY'))
        drivers = get_patient_shap_drivers(row['patient_id'])
        prompt  = (
            f"You are a clinical AI summarizing ICU patient risk for care teams. "
            f"Patient: {row['name']}, Age {row['age']}, Dx: {row['diagnosis']}. "
            f"Risk: {row['risk_score']:.0%} ({row['risk_level']}). "
            f"Top drivers: {', '.join(drivers)}. "
            f"Vitals: HR {row['hr']:.0f}, SpO2 {row['spo2']:.1f}%, SBP {row['sbp']:.0f}, RR {row['rr']:.0f}. "
            f"Labs: Lactate {row['lactate']:.1f}, WBC {row['wbc']:.1f}, Creatinine {row['creatinine']:.2f}. "
            f"NEWS2: {row['news2_score']}. "
            f"Write a concise 3-sentence clinical summary with specific recommended actions."
        )
        resp = client.chat.completions.create(
            model='nvidia/llama-3.1-nemotron-70b-instruct',
            messages=[{'role':'user','content':prompt}],
            max_tokens=200, temperature=0.3
        )
        return resp.choices[0].message.content

# ── Test on top 6 highest-risk patients ──────────────────────────────────────
print(f" Nemotron Clinical Insight Engine — {'MOCK' if MOCK_MODE else 'LIVE'} mode")
print("─" * 72)

top_pts = df.nlargest(6, 'risk_score')
for _, p in top_pts.iterrows():
    row = df[df['patient_id']==p['patient_id']].iloc[0]
    insight = generate_nemotron_insight(row)
    risk_color = "🔴" if row['risk_level']=='HIGH' else "🟡"
    print(f"{risk_color} {row['patient_id']}  |  {row['name']}  |  Age {int(row['age'])}  |  Risk: {row['risk_score']:.0%}  |  {row['diagnosis']}")
    print(f"   {textwrap.fill(insight, 70, subsequent_indent='   ')}")
    print()
print(" Nemotron insight engine verified.")

 Nemotron Clinical Insight Engine — MOCK mode
────────────────────────────────────────────────────────────────────────
🔴 ICU-0001  |  Noah Rhodes  |  Age 86  |  Risk: 100%  |  Pneumonia
   Noah (age 86, Pneumonia) presents with a 99% deterioration
   probability. Primary drivers are worsening SpO₂ Trend/hr and HR
   Trend/hr, with 7 concurrent clinical flags and NEWS2 score of 12.
   Recommend immediate attending review, blood cultures, and
   consideration of vasopressor initiation — estimated intervention
   window: 40–90 minutes.

🔴 ICU-0003  |  Daniel Wagner  |  Age 49  |  Risk: 100%  |  Post-cardiac surgery
   Daniel (age 49, Post-cardiac surgery) presents with a 99%
   deterioration probability. Primary drivers are worsening SpO₂
   Trend/hr and HR Trend/hr, with 7 concurrent clinical flags and
   NEWS2 score of 12. Recommend immediate attending review, blood
   cultures, and consideration of vasopressor initiation — estimated
   intervention window: 40–90 minutes.

🔴 ICU-0007  |

In [8]:
os.makedirs('../data',  exist_ok=True)
os.makedirs('../model', exist_ok=True)

# 1. Patient data with risk scores
df_export = df.copy()
df_export['risk_level'] = df_export['risk_level'].astype(str)
df_export.to_csv('../data/patients_with_risk.csv', index=False)
print(f" data/patients_with_risk.csv  ({len(df_export):,} rows, {len(df_export.columns)} cols)")

# 2. XGBoost model
model.save_model('../model/xgb_deterioration.json')
print(" model/xgb_deterioration.json")

# 3. Feature metadata
with open('../data/feature_cols.json','w') as f:
    json.dump({'feature_cols': FEATURE_COLS, 'feature_display': FEATURE_DISPLAY}, f, indent=2)
print(" data/feature_cols.json")

# 4. Ward hourly summary (simulated 24h)
hourly = pd.DataFrame({
    'hour':               list(range(24)),
    'high_risk_count':    [max(0, int(df[df['risk_level']=='HIGH'].shape[0]) + random.randint(-10,10)) for _ in range(24)],
    'moderate_risk_count':[max(0, int(df[df['risk_level']=='MODERATE'].shape[0]) + random.randint(-15,15)) for _ in range(24)],
    'alerts_fired':       [random.randint(0, 8) for _ in range(24)],
    'avg_risk_score':     [round(df['risk_score'].mean() + random.uniform(-0.05,0.05), 3) for _ in range(24)],
    'interventions':      [random.randint(0, 4) for _ in range(24)],
})
hourly.to_csv('../data/ward_hourly.csv', index=False)
print(" data/ward_hourly.csv")

# 5. Alert queue (top 20 patients for dashboard)
alert_queue = df.nlargest(20,'risk_score')[
    ['patient_id','name','age','gender','diagnosis','room',
     'hr','spo2','sbp','rr','risk_score','risk_level','news2_score',
     'combined_flag_count','trend_severity','alert']
].copy()
alert_queue['risk_level'] = alert_queue['risk_level'].astype(str)
alert_queue.to_csv('../data/alert_queue.csv', index=False)
print(" data/alert_queue.csv")

print("\n All outputs saved. Launch the dashboard with:")
print("   conda activate nvidia-health-demo")
print("   python dashboard/app.py")

 data/patients_with_risk.csv  (847 rows, 39 cols)
 model/xgb_deterioration.json
 data/feature_cols.json
 data/ward_hourly.csv
 data/alert_queue.csv

 All outputs saved. Launch the dashboard with:
   conda activate nvidia-health-demo
   python dashboard/app.py
